### Ingestion del archivo "language_role.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
languagerole_schema = StructType(fields=[
    StructField("roleId", IntegerType(), True),
    StructField("languageRole", StringType(), True)
    ])

In [0]:
languagerole_df = spark.read \
            .schema(languagerole_schema) \
            .option("multiLine", "true") \
            .json(f"{bronze_folder_path}/{v_file_date}/language_role.json")

In [0]:
display(languagerole_df)

#### Paso 2 - renombrar columnas y agregar columnas nuevas
1. renombrar "roleId" a "role_id"
2. renombrar "languageRole" a "language_role"
3. Agregar las columnas "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
languagerole_with_columns_df = add_ingestion_date(languagerole_df) \
                            .withColumnsRenamed({"roleId": "role_id"}) \
                            .withColumnsRenamed({"languageRole": "language_role"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(languagerole_with_columns_df)

#### Paso 4 - Escribir salida en un formato "Parquet"

In [0]:
# languagerole_with_columns_df.write.mode("overwrite").parquet(f"{silver_folder_path}/languages_roles")

In [0]:
# display(spark.read.parquet("abfss://silver@moviehistory22.dfs.core.windows.net/languages_roles"))

In [0]:
languagerole_with_columns_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages_roles")

In [0]:
%sql
SELECT * FROM movie_silver.languages_roles

In [0]:
dbutils.notebook.exit("Success")